<a href="https://colab.research.google.com/github/ancestor9/Listeria/blob/main/Listeria_05_Extract_Data_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **LIMS 데이터 바탕으로 Stacking Model 적용**
>#### **1. LIMS 데이터 전처리 (검출 품목만을 대상으로 자료 구성)**
>#### **2. Optuna 모듈로 Hyperparameter**
>#### **3. Stacking Model 적용**


<font color='red'> 아래 코드는 전체 데이터의 2.5% 만 돌린 것임 (GPU가 없어서 !!)

> Further Study(앞으로 할 일)

>> 불검출/검출 데이터를 포함한 전체 데이터 구성

>> 기후 시계열자료와 수질자료 등을 포함하여 데이터 구성

>> Deep Learning 모델 적용

>> 60일 미레 예측은 본부에서 최근 5년 동안 시계열예측을 통해 배포 받은 자료를 적용

In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
import pandas as pd
path ='/content/drive/MyDrive/MFDS/'
df = pd.read_parquet(path + "listeria_dataset.parquet")
print(len(df))

676531


In [73]:
sf= df.copy()
df = sf.sample(frac=0.025)
print(len(df))

16913


In [74]:
def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)

In [75]:
summarize_df(df).to_excel('/content/listeria_dataset_summary.xlsx', index=False)

In [76]:
selection_list = pd.read_excel(path + "listeria_dataset_summary_selection.xlsx")
selection_list

,Column,feature 포함여부,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,lims_data_key,0,676531,string,0,0.00,"['O002012_250100044_1_5', 'O000728_250103704_4..."
1,hazard_label,0,1,category,0,0.00,['리스테리아']
2,judge_result,1,2,category,0,0.00,"['불검출', '검출']"
3,src_route,1,2,category,0,0.00,"['domestic', 'overseas']"
4,data_source,0,1,category,0,0.00,['식약처']
5,expr_iem_nm,0,4,category,0,0.00,"['리스테리아모노사이토제네스', '리스테리아 모노사이토제네스', '리스테리아 모노싸..."
6,result_value,0,100,string,0,0.00,"['음성,음성,음성,음성,음성', '음성', '불검출', '음성, 음성, 음성, 음..."
7,result_value_num,0,2,float32,672754,99.44,"[np.float32(0.0), np.float32(2.0)]"
8,judge_stdr,0,17,string,669258,98.92,"['n=5,c=0,m=0/25g', 'n=5, c=0, m=0/25g', '음성 ..."
9,judge_stdr_num,0,5,float32,676463,99.99,"[np.float32(100.0), np.float32(20.0), np.float..."


In [77]:
selection_list[selection_list['feature 포함여부'] == 1]['Column'].unique()

array(['judge_result', 'src_route', 'inspct_instt_code',
       'train_prdlst_nm', 'train_prdlst_nm_2', 'prduct_nm', 'mnfctur_de',
       'addr', 'lat', 'lon'], dtype=object)

In [78]:
df = df[selection_list[selection_list['feature 포함여부'] == 1]['Column'].unique()].copy()
df

,judge_result,src_route,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,addr,lat,lon
226542,불검출,domestic,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,경기도 파주시 조리읍 등원로330번길 110,37.744717,126.805168
97484,불검출,domestic,O000124,유가공품류,우유류,강성원우유,2018-06-12,충청남도 예산군 가마시로 75,36.680882,126.844849
359135,불검출,domestic,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,충청남도 금산군 군북면 군북로 586,36.168907,127.527725
387695,불검출,domestic,O000146,유가공품류,가공유류,소화가 잘되는 우유 오리지널,2022-11-22,광주광역시 광산구 어등대로 511 (운수동),35.162342,126.773270
196336,불검출,domestic,O000098,조리식품 등,조리식품 등,호박전,2022-01-10,대전광역시 서구 괴정동 90-16,36.337139,127.382065
...,...,...,...,...,...,...,...,...,...,...
177175,불검출,domestic,O000858,유가공품류,가공유류,미숫가루,2023-09-06,경기도 평택시 산단로 106 (모곡동),37.034405,127.075668
454317,불검출,domestic,O000728,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),2025-04-16,전라남도 함평군 학교면 학교길 281-22,35.016914,126.537537
164588,불검출,domestic,O000066,기타기준규격외,기타기준규격외,농협 홍삼진액 완제품(살균 후),2021-12-16,경상북도 군위군 의흥면 동부로 935-14,36.175896,128.715363
330382,불검출,domestic,O000044,유가공품류,치즈류,펠렛치즈 이코노Ⅰ,2018-02-19,경상북도 김천시 어모면 산업단지6로 140,36.180428,128.118729


### 의문사항
- result_value의 "음성,음성,음성,음성,음성" 의미는?
- expr_iem_nm, legal_dong_code(47641개 결측치는 모두 해외임), legal_dong_code(3772개)는 시군단위 하위 지역정보 로 제거

In [79]:
df['judge_result'].value_counts(normalize=True)*100

,proportion
judge_result,
불검출,99.686632
검출,0.313368


In [80]:

df['src_route'].value_counts()

,count
src_route,
domestic,15673
overseas,1240


## 검출된 데이터만 가지고 학습(가정)
- 즉 과거의 불검출된 품목은 모두 위험이 없다고 가정함
> - **품목기준 : train_prdlst_code_2    하위 품목군 코드 = 품목군_3단계**

In [81]:
# @title 식품종류별로 검사결과 검출이 된 품목과 검측률
df['judge_result'] = df['judge_result'].map({'검출': 1, '불검출': 0}).astype(int)
product_list_tm = df.groupby('train_prdlst_nm_2')['judge_result'].mean().sort_values(ascending=False)*100
product_list_tm = product_list_tm[product_list_tm != 0]
product_list_tm

/tmp/ipykernel_923/2712643283.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  product_list_tm = df.groupby('train_prdlst_nm_2')['judge_result'].mean().sort_values(ascending=False)*100


,judge_result
train_prdlst_nm_2,
포장육,5.333333
기타,2.266289
해양어류,1.282051
수산물,0.854701
기타기준규격외,0.344590
...,...
환경오염도조사용,NaN
회유어류,NaN
효모식품,NaN


In [82]:
print(f'검출율이 "0"이 아닌 식품종류 개수는 {len(product_list_tm)} 종류')

검출율이 "0"이 아닌 식품종류 개수는 108 종류


In [83]:
# @title 리스테리아가 검출된 품목만 선택하여 데이터 구성
product_list_non_tm = product_list_tm[product_list_tm == 0]
print(product_list_non_tm.index)
tf = df[df['train_prdlst_nm_2'].isin(product_list_tm.index)].copy()
print("tf['train_prdlst_nm_2'] 열(Column)에 들어있는 상품명 중 product_list_non_tm.index에 존재하는 상품명이 단 하나라도 있는가?")
tf['train_prdlst_nm_2'].isin(product_list_non_tm.index).any()

CategoricalIndex([], categories=['NAG(엔에이지, N-아세틸글루코사민, N-Acetylglucosamine)', '가공김(조미김 또는 구운김)', '가공두부', '가공유류', ..., '효모식품', '효모추출물', '효소식품', '효소처리스테비아'], ordered=False, dtype='category', name='train_prdlst_nm_2')
tf['train_prdlst_nm_2'] 열(Column)에 들어있는 상품명 중 product_list_non_tm.index에 존재하는 상품명이 단 하나라도 있는가?


np.False_

In [84]:
# @title 식품종류별로 수입 및 국내를 구분하여 검출룰을 판단
detection_rate_by_product_route = (
    tf.groupby(['train_prdlst_nm_2', 'src_route'], observed=True)['judge_result']
    .mean()
    .unstack('src_route')
    .fillna(0) * 100
)

# domestic(국내) 수치 기준으로 내림차순 정렬하여 출력
detection_rate_by_product_route.sort_values(by='domestic', ascending=False)

src_route,domestic,overseas
train_prdlst_nm_2,,
포장육,5.333333,0.000000
기타,2.266289,0.000000
기타기준규격외,0.344637,0.000000
소시지류,0.261780,0.000000
즉석섭취.편의식품류,0.204082,0.000000
수산물,0.000000,0.854701
유가공품류,0.000000,0.492611
양념육류,0.000000,2.325581
해양어류,0.000000,1.351351


# 2026_10_7

1. 예측 기간 변경

- 기존: 11일 예측

- 변경: 60일 예측



2. 예측 결과 단위 변경

- 기존: 검체 단위 예측

- 변경: 지역 × 품목군_2단계 × 품목군_3단계 단위로 결과 산출

  (단, 가공식품의 경우 품목군_3단계 × 품목군_4단계)



같은 지역·같은 품목군이라도 하위 품목군이 다르면 각각 별도의 예측 단위로 구분됩니다.



예를 들어 "경기도 안성시 × 견과종실류"만으로는 하나의 단위가 아니고,

"경기도 안성시 × 견과종실류(2단계) × 땅콩 또는 견과류(3단계)"와

"경기도 안성시 × 견과종실류(2단계) × 유지종실류(3단계)"가 각각 별개의 예측 단위가 되어 확률도 따로 산출됩니다.


■ 추가 컬럼

> train_prdlst_code_2    하위 품목군 코드 = 품목군_3단계

>  train_prdlst_nm_2      하위 품목군명



- 기존 품목군 = 2단계(train_prdlst_code)보다 한 단계 아래의 분류입니다.

  하위 단계가 없는 품목은 기존 품목군 값을 그대로 넣었습니다.

>  (단, 가공식품의 경우 품목군_3단계 × 품목군_4단계) 가공식품은 코드가 C로 시작하는 품목입니다.

## **데이터 전처리**

### **1. addr(주소) 데이터 전처리**

- 지역정보 feature를 위해 해외데이터는 제거

In [85]:
tf['src_route'].value_counts()

,count
src_route,
domestic,11178
overseas,537


In [86]:
tf = tf[tf['src_route'] == 'domestic'].copy()
print(len(tf))

11178


In [87]:
# 'addr' 컬럼의 문자열을 공백 기준으로 분할하여 '시도'와 '시군구'를 추출합니다.
# 데이터에 결측치나 비정상적인 주소가 있을 수 있으므로 string 타입으로 변환 후 처리합니다.
tf['sido'] = tf['addr'].astype(str).apply(lambda x: x.split()[0] if len(x.split()) > 0 else '')
tf['sigungu'] = tf['addr'].astype(str).apply(lambda x: x.split()[1] if len(x.split()) > 1 else '')
del tf['addr']
del tf['src_route']
tf

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구
...,...,...,...,...,...,...,...,...,...,...
626433,0,O000001,즉석식품류,즉석섭취.편의식품류,한나도시락,2017-09-12,37.305672,127.009613,경기도,수원시
443916,0,O000858,기타기준규격외,기타기준규격외,액젓-후살균 후,2025-05-20,35.407345,126.879517,전라남도,장성군
307489,0,O000001,식육가공품 및 포장육,소시지류,치즈킬바사,2025-12-05,37.889027,127.217720,경기도,포천시
454317,0,O000728,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),2025-04-16,35.016914,126.537537,전라남도,함평군


In [88]:
tf

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구
...,...,...,...,...,...,...,...,...,...,...
626433,0,O000001,즉석식품류,즉석섭취.편의식품류,한나도시락,2017-09-12,37.305672,127.009613,경기도,수원시
443916,0,O000858,기타기준규격외,기타기준규격외,액젓-후살균 후,2025-05-20,35.407345,126.879517,전라남도,장성군
307489,0,O000001,식육가공품 및 포장육,소시지류,치즈킬바사,2025-12-05,37.889027,127.217720,경기도,포천시
454317,0,O000728,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),2025-04-16,35.016914,126.537537,전라남도,함평군


### **3. 품목군별 Target Mean 적용**
- 데이터가 충분히 많기 떄문에 과거 부적합 비율을 사용해도 무방하다고 판단(Data Leakage는 없다고 판단)

In [89]:
# train_prdlst_nm_2 단위로 judge_result의 평균값 계산하여 새로운 컬럼으로 추가
target_means = tf.groupby('train_prdlst_nm_2')['judge_result'].mean()
tf['target_mean_by_product_2'] = tf['train_prdlst_nm_2'].map(target_means)
tf[['train_prdlst_nm_2', 'prduct_nm', 'target_mean_by_product_2']].head()

/tmp/ipykernel_923/53235399.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_means = tf.groupby('train_prdlst_nm_2')['judge_result'].mean()


,train_prdlst_nm_2,prduct_nm,target_mean_by_product_2
226542,기타기준규격외,편육 1차냉각,0.003446
359135,기타기준규격외,산삼배양근농축액,0.003446
611347,기타기준규격외,절임식품(꼬막)-세척후,0.003446
175043,기타기준규격외,도시락(냉장운반),0.003446
183194,기타기준규격외,1907ASA-11,0.003446


### **3. 대한 민국을 pixel로 만들어 지역정보를 구분(위도 경도 정보 --> 격자 Grid)**
- 대한민국 정도의 좁은 지역에서는 위도·경도를 다음처럼 근사

> 위도 1° ≈ 111 km

> 경도 1° ≈ 88~92 km 정도

- **make_grid() 함수로 km 단위로 격자 셀을 생성하여 시계열정보와 연계하여 예측 가능**

> **10km, 30km, 50km 등등 Hyperparameter Tunning 실행**

> **기후정보와 join시 위도와 경도가 같은 grid 이면 Join하여 사용**


In [90]:
def make_grid(df, km):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Lon'], df['Lat']),
        crs='EPSG:4326'
    )

    gdf = gdf.to_crs('EPSG:5179')

    grid_size = km * 1000

    gdf['grid_x'] = (gdf.geometry.x // grid_size).astype(int)
    gdf['grid_y'] = (gdf.geometry.y // grid_size).astype(int)

    gdf['Grid_ID'] = (
        gdf['grid_x'].astype(str) + '_' +
        gdf['grid_y'].astype(str)
    )

    return gdf

In [91]:
import geopandas as gpd

# Lon, Lat을 소문자 lon, lat 컬럼명에 맞춰 적용되도록 make_grid 함수 재정의 및 실행
def make_grid_fixed(df, km):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['lon'], df['lat']),
        crs='EPSG:4326'
    )
    gdf = gdf.to_crs('EPSG:5179')
    grid_size = km * 1000
    gdf['grid_x'] = (gdf.geometry.x // grid_size).astype(int)
    gdf['grid_y'] = (gdf.geometry.y // grid_size).astype(int)
    gdf['Grid_ID'] = (
        gdf['grid_x'].astype(str) + '_'.encode().decode() +
        gdf['grid_y'].astype(str)
    )
    return gdf


In [92]:
# @title 10 km × 10 km 크기의 격자 중에서 97번째 × 188번째 위치
tf_grid = make_grid_fixed(tf, 10) # make_grid_fixed(tf, 20)
tf_grid.head()

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,geometry,grid_x,grid_y,Grid_ID
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시,0.003446,POINT (938784.375 1971903.622),93,197,93_197
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군,0.003446,POINT (1002493.469 1796868.328),100,179,100_179
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면,0.003446,POINT (972838.726 1853757.621),97,185,97_185
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군,0.003446,POINT (969027.438 1765257.39),96,176,96_176
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구,0.003446,POINT (943682.7 1949450.892),94,194,94_194


In [93]:
# drop() 메소드를 사용하여 여러 컬럼을 한 번에 삭제합니다.
tf_grid = tf_grid.drop(columns=['Grid_ID', 'geometry'])
tf_grid.head()

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,grid_x,grid_y
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시,0.003446,93,197
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군,0.003446,100,179
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면,0.003446,97,185
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군,0.003446,96,176
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구,0.003446,94,194


In [94]:
tf_grid

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,grid_x,grid_y
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시,0.003446,93,197
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군,0.003446,100,179
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면,0.003446,97,185
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군,0.003446,96,176
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구,0.003446,94,194
...,...,...,...,...,...,...,...,...,...,...,...,...,...
626433,0,O000001,즉석식품류,즉석섭취.편의식품류,한나도시락,2017-09-12,37.305672,127.009613,경기도,수원시,0.002041,95,192
443916,0,O000858,기타기준규격외,기타기준규격외,액젓-후살균 후,2025-05-20,35.407345,126.879517,전라남도,장성군,0.003446,94,171
307489,0,O000001,식육가공품 및 포장육,소시지류,치즈킬바사,2025-12-05,37.889027,127.217720,경기도,포천시,0.002618,97,198
454317,0,O000728,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),2025-04-16,35.016914,126.537537,전라남도,함평군,0.003446,91,166


### **4. 제품명에 대한 text mining으로 feature 생성**
- LLM에게 제품명에 대해 부적합비율과 관련성이 높은 text를 clustering prompt

In [95]:
tf_grid[['train_prdlst_nm', 'train_prdlst_nm_2', 'prduct_nm', 'target_mean_by_product_2']]

,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,target_mean_by_product_2
226542,기타기준규격외,기타기준규격외,편육 1차냉각,0.003446
359135,기타기준규격외,기타기준규격외,산삼배양근농축액,0.003446
611347,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,0.003446
175043,기타기준규격외,기타기준규격외,도시락(냉장운반),0.003446
183194,기타기준규격외,기타기준규격외,1907ASA-11,0.003446
...,...,...,...,...
626433,즉석식품류,즉석섭취.편의식품류,한나도시락,0.002041
443916,기타기준규격외,기타기준규격외,액젓-후살균 후,0.003446
307489,식육가공품 및 포장육,소시지류,치즈킬바사,0.002618
454317,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),0.003446


In [96]:
df = tf_grid.copy()

In [97]:
import re

# 2. 리스테리아 특화 파생 변수 추출 함수
def extract_listeria_features(text):
    text_str = str(text).strip()
    text_lower = text_str.lower()

    # --- A. 가열 / 살균 상태 (heat_process) ---
    heat_process = '기타/공정중'
    if re.search(r'살균후|가열후|구이후|볶음후|증자후|멸균|훈연|훈제', text_str):
        heat_process = '가열_살균후'
    elif re.search(
        r'살균전|가열전|구이전|생식|비가열|세척후|전처리후|원재료|절단', text_str
    ):
        heat_process = '가열전_비가열_원재료'
    elif re.search(r'ccp', text_lower):
        heat_process = 'CCP공정'

    # --- B. 공정 및 완제품 단계 (process_stage) ---
    process_stage = '공정중'
    if re.search(
        r'완제품|포장후|바로섭취|도시락|샌드위치|김밥|샐러드|마들렌|프레첼|강정',
        text_str,
    ):
        process_stage = '완제품_RTE가능'
    elif re.search(r'원재료|원료|세척|전처리|절단|성형', text_str):
        process_stage = '원재료_전처리'

    # --- C. 보관 및 제형 특성 (storage_form) ---
    storage_form = '상온/일반'
    if re.search(r'냉장|슬라이스|해동|훈제|훈연|습윤', text_str):
        storage_form = '냉장_습윤'
    elif re.search(r'냉동', text_str):
        storage_form = '냉동'
    elif re.search(r'동결건조|건조|분말|가루|입자|멸균', text_str):
        storage_form = '건조_분말_멸균'

    # --- D. 리스테리아 고위험 개별 지표 (Binary) ---
    # RTE(즉석섭취)/훈제/유제품/가공육/샐러드/샌드위치 등 리스테리아 고위험 키워드
    is_rte_risk = int(
        bool(
            re.search(
                r'샌드위치|김밥|샐러드|훈제|훈연|슬라이스|치즈|유제품|참치마요|도시락|즉석|바로|완제품',
                text_str,
            )
        )
    )

    # 냉장/해동/세척후 등 습윤 상태
    is_chilled_wet = int(
        bool(re.search(r'냉장|해동|세척후|생식|습윤', text_str))
    )

    # --- E. 교차 변수 (Interaction Terms) ---
    # 1) 가열/살균 상태 x 공정 단계
    heat_X_stage = f'{heat_process}_X_{process_stage}'

    # 2) 가열/살균 상태 x 보관/제형 특성
    heat_X_storage = f'{heat_process}_X_{storage_form}'

    # 3) RTE 고위험 여부 x 냉장/습윤 여부 (1이면 최우선 리스테리아 고위험군)
    rte_X_chilled = is_rte_risk * is_chilled_wet

    return pd.Series([
        heat_process,
        process_stage,
        storage_form,
        is_rte_risk,
        is_chilled_wet,
        heat_X_stage,
        heat_X_storage,
        rte_X_chilled,
    ])


# 3. 데이터프레임 변수 적용
feature_cols = [
    'heat_process',
    'process_stage',
    'storage_form',
    'is_rte_risk',
    'is_chilled_wet',
    'heat_X_stage',
    'heat_X_storage',
    'rte_X_chilled',
]

In [98]:
df[feature_cols] = df['prduct_nm'].apply(extract_listeria_features)
df

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,...,grid_x,grid_y,heat_process,process_stage,storage_form,is_rte_risk,is_chilled_wet,heat_X_stage,heat_X_storage,rte_X_chilled
226542,0,O000169,기타기준규격외,기타기준규격외,편육 1차냉각,2020-07-09,37.744717,126.805168,경기도,파주시,...,93,197,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
359135,0,O000728,기타기준규격외,기타기준규격외,산삼배양근농축액,2023-08-22,36.168907,127.527725,충청남도,금산군,...,100,179,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
611347,0,O000858,기타기준규격외,기타기준규격외,절임식품(꼬막)-세척후,2023-11-08,36.681396,127.195999,세종특별자치시,전의면,...,97,185,가열전_비가열_원재료,원재료_전처리,상온/일반,0,1,가열전_비가열_원재료_X_원재료_전처리,가열전_비가열_원재료_X_상온/일반,0
175043,0,O000858,기타기준규격외,기타기준규격외,도시락(냉장운반),2021-11-10,35.883419,127.156853,전라북도,완주군,...,96,176,기타/공정중,완제품_RTE가능,냉장_습윤,1,1,기타/공정중_X_완제품_RTE가능,기타/공정중_X_냉장_습윤,1
183194,0,O000039,기타기준규격외,기타기준규격외,1907ASA-11,2019-07-26,37.542667,126.862495,서울특별시,강서구,...,94,194,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626433,0,O000001,즉석식품류,즉석섭취.편의식품류,한나도시락,2017-09-12,37.305672,127.009613,경기도,수원시,...,95,192,기타/공정중,완제품_RTE가능,상온/일반,1,0,기타/공정중_X_완제품_RTE가능,기타/공정중_X_상온/일반,0
443916,0,O000858,기타기준규격외,기타기준규격외,액젓-후살균 후,2025-05-20,35.407345,126.879517,전라남도,장성군,...,94,171,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
307489,0,O000001,식육가공품 및 포장육,소시지류,치즈킬바사,2025-12-05,37.889027,127.217720,경기도,포천시,...,97,198,기타/공정중,공정중,상온/일반,1,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
454317,0,O000728,기타기준규격외,기타기준규격외,한우양지(냉장)포장육(완제품),2025-04-16,35.016914,126.537537,전라남도,함평군,...,91,166,기타/공정중,완제품_RTE가능,냉장_습윤,1,1,기타/공정중_X_완제품_RTE가능,기타/공정중_X_냉장_습윤,1


### **5. prduct_nm의 전체 텍스트에 대해 Char/Word N-gram 기반 TF-IDF Vectorizer를 적용**
- TF-IDF + 머신러닝/딥러닝 연계: TF-IDF Vectorizer를 적용한 후 LightGBM, XGBoost 또는 TabNet 모델에 함께 입력값으로 넣어주면 텍스트의 정교한 패턴까지 학습에 활용

In [99]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 모델 불러오기
import lightgbm as lgb
from xgboost import XGBClassifier

In [100]:
# ---------------------------------------------------------
# 1. 타겟 변수(y) 및 입력 데이터(X) 준비
# ---------------------------------------------------------
TARGET_COL = 'judge_result'

# Train / Test Split (Stratify 적용)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df, df[TARGET_COL], test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)

# ---------------------------------------------------------
# 2. TF-IDF 피처 추출 (Char N-gram + Word N-gram)
# ---------------------------------------------------------
# 결측치(Null/NA) 처리: 결측치를 빈 문자열으로 변환하여 TfidfVectorizer의 NAType 오류 방지
X_train_text = X_train_df['prduct_nm'].fillna('').astype(str)
X_test_text = X_test_df['prduct_nm'].fillna('').astype(str)

# Char N-gram: 글자 단위 (2~4자 연쇄) -> 띄어쓰기 오차, 붙여쓴 단어에 강함
tfidf_char = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 4), max_features=300, min_df=5
)

# Word N-gram: 단어 단위 (1~2단어 조합)
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=200, min_df=3
)

# Fit & Transform
X_train_char = tfidf_char.fit_transform(X_train_text)
X_train_word = tfidf_word.fit_transform(X_train_text)

X_test_char = tfidf_char.transform(X_test_text)
X_test_word = tfidf_word.transform(X_test_text)

# ---------------------------------------------------------
# 3. 규칙 기반(Rule-based) 생성 변수 인코딩 및 스케일링
# ---------------------------------------------------------
cat_cols = [
    'heat_process',
    'process_stage',
    'storage_form',
    'heat_X_stage',
    'heat_X_storage',
]
num_cols = ['is_rte_risk', 'is_chilled_wet', 'rte_X_chilled']

# 범주형 -> One-Hot Encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(X_train_df[cat_cols])
X_test_cat = ohe.transform(X_test_df[cat_cols])

# 수치형 -> StandardScaler
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_df[num_cols])
X_test_num = scaler.transform(X_test_df[num_cols])

# ---------------------------------------------------------
# 4. 희소 행렬(Sparse Matrix) 하나로 결합
# ---------------------------------------------------------
# [규칙 범주형(OHE) + 규칙 수치형 + TF-IDF Char + TF-IDF Word]
X_train_final = hstack([
    X_train_cat,
    csr_matrix(X_train_num),
    X_train_char,
    X_train_word,
]).tocsr()

X_test_final = hstack([
    X_test_cat,
    csr_matrix(X_test_num),
    X_test_char,
    X_test_word,
]).tocsr()

print(f'최종 Train 피처 수: {X_train_final.shape[1]}개')
print(f'최종 Test 피처 수: {X_test_final.shape[1]}개')


최종 Train 피처 수: 542개
최종 Test 피처 수: 542개


### **6. 전체 모델 평가**

In [101]:
!pip install catboost -q

In [104]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Gradient Boosting 계열 모델
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from tqdm import tqdm

In [105]:
# ---------------------------------------------------------
# 3. 주요 모델 정의 (불균형 데이터 보정 포함)
# ---------------------------------------------------------
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=7,
        class_weight='balanced',
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=ratio,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        verbose=0,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1
    ),
        'Extra Trees': ExtraTreesClassifier(
        n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
}

# ---------------------------------------------------------
# 4. 모델별 학습, 평가 및 오탐/PR-AUC 지표 계산
# ---------------------------------------------------------
results = []

for name, model in tqdm(models.items()):
    print(f'\n=== {name} 모델 학습 및 평가 시작 ===')

    # 학습
    model.fit(X_train_final, y_train)

    # 예측 (확률 및 클래스)
    pred_prob = model.predict_proba(X_test_final)[:, 1]
    pred = (pred_prob >= 0.5).astype(int)

    # 1) PR-AUC 계산
    precision_vec, recall_vec, _ = precision_recall_curve(y_test, pred_prob)
    pr_auc_val = auc(recall_vec, precision_vec)

    # 2) Confusion Matrix 오탐지표 계산 (FN, FP)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_errors = fn + fp
    error_rate = (total_errors / len(y_test)) * 100

    # 3) 기타 평가 지표
    roc_auc_val = roc_auc_score(y_test, pred_prob)
    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f'[{name} 평가 결과]')
    print(f'PR-AUC Score        : {pr_auc_val:.4f}')
    print(f'ROC-AUC Score       : {roc_auc_val:.4f}')
    print(f'F1-Score            : {f1_val:.4f}')
    print(
        f'총 오류 건수(FN+FP)  : {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP: {fp}건)'
    )
    print(f'오탐 비율(%)        : {error_rate:.2f}%')
    print('\n[Classification Report]')
    print(classification_report(y_test, pred, zero_division=0))

    results.append({
        'Model': name,
        'PR-AUC': pr_auc_val,
        '오류건수(FN+FP)': total_errors,
        '오탐비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp,
        'ROC-AUC': roc_auc_val,
        'F1-Score': f1_val,
        'Recall': rec_val,
        'Precision': prec_val,
    })

# ---------------------------------------------------------
# 5. 모델 비교 요약 표 출력
# ---------------------------------------------------------
summary_df = (
    pd.DataFrame(results)
    .sort_values(by='PR-AUC', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 60)
print('[전체 모델 종합 성능 비교 요약 (PR-AUC 순 정렬)]')
print('=' * 60)
print(summary_df.to_string(index=False))

  0%|          | 0/6 [00:00<?, ?it/s]


=== LightGBM 모델 학습 및 평가 시작 ===


 17%|█▋        | 1/6 [00:03<00:16,  3.35s/it]

[LightGBM 평가 결과]
PR-AUC Score        : 0.0875
ROC-AUC Score       : 0.8071
F1-Score            : 0.1053
총 오류 건수(FN+FP)  : 68건 (미검출 FN: 6건, 과잉검출 FP: 62건)
오탐 비율(%)        : 3.04%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      2226
           1       0.06      0.40      0.11        10

    accuracy                           0.97      2236
   macro avg       0.53      0.69      0.54      2236
weighted avg       0.99      0.97      0.98      2236


=== XGBoost 모델 학습 및 평가 시작 ===


 33%|███▎      | 2/6 [00:07<00:14,  3.55s/it]

...

[CatBoost 평가 결과]
PR-AUC Score        : 0.0846
ROC-AUC Score       : 0.7191
F1-Score            : 0.1212
총 오류 건수(FN+FP)  : 58건 (미검출 FN: 6건, 과잉검출 FP: 52건)
오탐 비율(%)        : 2.59%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      2226
           1       0.07      0.40      0.12        10

    accuracy                           0.97      2236
   macro avg       0.53      0.69      0.55      2236
weighted avg       0.99      0.97      0.98      2236


=== Random Forest 모델 학습 및 평가 시작 ===


 67%|██████▋   | 4/6 [00:18<00:09,  4.62s/it]

[Random Forest 평가 결과]
PR-AUC Score        : 0.3175
ROC-AUC Score       : 0.6147
F1-Score            : 0.1905
총 오류 건수(FN+FP)  : 34건 (미검출 FN: 6건, 과잉검출 FP: 28건)
오탐 비율(%)        : 1.52%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2226
           1       0.12      0.40      0.19        10

    accuracy                           0.98      2236
   macro avg       0.56      0.69      0.59      2236
weighted avg       0.99      0.98      0.99      2236


=== Extra Trees 모델 학습 및 평가 시작 ===


100%|██████████| 6/6 [00:20<00:00,  3.44s/it]

[Extra Trees 평가 결과]
PR-AUC Score        : 0.0708
ROC-AUC Score       : 0.6595
F1-Score            : 0.1905
총 오류 건수(FN+FP)  : 34건 (미검출 FN: 6건, 과잉검출 FP: 28건)
오탐 비율(%)        : 1.52%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2226
           1       0.12      0.40      0.19        10

    accuracy                           0.98      2236
   macro avg       0.56      0.69      0.59      2236
weighted avg       0.99      0.98      0.99      2236


=== Logistic Regression 모델 학습 및 평가 시작 ===
[Logistic Regression 평가 결과]
PR-AUC Score        : 0.0728
ROC-AUC Score       : 0.7621
F1-Score            : 0.0641
총 오류 건수(FN+FP)  : 146건 (미검출 FN: 5건, 과잉검출 FP: 141건)
오탐 비율(%)        : 6.53%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      2226
           1       0.03      0.50      0.06        10

    accuracy                           0.93   

In [106]:
summary_df

,Model,PR-AUC,오류건수(FN+FP),오탐비율(%),미검출(FN),과잉검출(FP),ROC-AUC,F1-Score,Recall,Precision
0,Random Forest,0.317517,34,1.520572,6,28,0.614690,0.190476,0.4,0.125000
1,LightGBM,0.087490,68,3.041145,6,62,0.807098,0.105263,0.4,0.060606
2,CatBoost,0.084639,58,2.593918,6,52,0.719115,0.121212,0.4,0.071429
3,XGBoost,0.082693,46,2.057245,6,40,0.736343,0.148148,0.4,0.090909
4,Logistic Regression,0.072836,146,6.529517,5,141,0.762129,0.064103,0.5,0.034247
5,Extra Trees,0.070763,34,1.520572,6,28,0.659524,0.190476,0.4,0.125000


### **7. 데이터 특성 번영 전체 모델 평가(휴리스틱 수동 설정)**
- 불균형 가중치 조정: 전체 ratio를 그대로 쓰면 과잉검출(FP)이 수천 건 급증하므로, scale_pos_weight 지수를 완화 조율
- 모델 복잡도 제한: max_depth, num_leaves, min_child_samples, min_samples_leaf 등을 통해 트리 깊이를 제한하여 FP 발생을 억제
- F1-Score 기반 임계값(Threshold) 자동 탐색: 고정 $0.5$ 임계값 대신 Precision-Recall 커브에서 F1-Score가 최대가 되는 최적의 임계값을 찾아 분류 기준으로 적용

In [107]:
# ---------------------------------------------------------
# 1. 임계값 탐색용 검증셋 분리 + 불균형 가중치 설정
# ---------------------------------------------------------
# 임계값은 검증셋에서만 찾고, 테스트셋은 최종 평가에만 사용 (누수 방지)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_final, y_train,
    test_size=0.2, stratify=y_train, random_state=42,
)

num_pos = int(np.sum(y_fit))
num_neg = len(y_fit) - num_pos

# 전체 ratio의 25%로 완화하여 과잉검출(FP) 억제
balanced_ratio = (num_neg / num_pos) * 0.25

# ---------------------------------------------------------
# 2. 주요 모델 정의 (가중치 완화 + 복잡도 제어)
# ---------------------------------------------------------
models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=5,
        num_leaves=20,
        min_child_samples=30,
        scale_pos_weight=balanced_ratio,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=balanced_ratio,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    ),
    'CatBoost': CatBoostClassifier(
        iterations=200,
        learning_rate=0.03,
        depth=5,
        scale_pos_weight=balanced_ratio,
        random_state=42,
        verbose=0,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=10,
        class_weight={0: 1, 1: 5},
        random_state=42,
        n_jobs=-1,
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=10,
        class_weight={0: 1, 1: 5},
        random_state=42,
        n_jobs=-1,
    ),
    'Logistic Regression': LogisticRegression(
        class_weight={0: 1, 1: 5},
        C=0.1,
        max_iter=1000,
        random_state=42,
    ),
}


# ---------------------------------------------------------
# 3. 검증셋 기준 F1 최적 임계값 탐색 함수
# ---------------------------------------------------------
def find_best_threshold(y_true, prob):
    p, r, th = precision_recall_curve(y_true, prob)
    denom = p + r
    f1 = np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)
    # p, r는 th보다 길이가 1 길고 마지막 원소는 (1, 0)이므로 th 범위로 제한
    best_idx = np.argmax(f1[:-1])
    return th[best_idx], f1[best_idx]


# ---------------------------------------------------------
# 4. 모델별 학습, 임계값 탐색(검증셋), 평가(테스트셋)
# ---------------------------------------------------------
results = []

for name, model in tqdm(models.items()):
    print(f'\n=== {name} 모델 학습 및 평가 시작 ===')

    # 학습 (fit 구간만 사용)
    model.fit(X_fit, y_fit)

    # 1) 검증셋에서 최적 임계값 결정
    val_prob = model.predict_proba(X_val)[:, 1]
    best_threshold, val_f1 = find_best_threshold(y_val, val_prob)

    # 2) 테스트셋 예측: 검증셋에서 정한 임계값을 그대로 적용
    pred_prob = model.predict_proba(X_test_final)[:, 1]
    pred = (pred_prob >= best_threshold).astype(int)

    # 3) 평가 지표
    precision_vec, recall_vec, _ = precision_recall_curve(y_test, pred_prob)
    pr_auc_val = auc(recall_vec, precision_vec)
    ap_val = average_precision_score(y_test, pred_prob)
    roc_auc_val = roc_auc_score(y_test, pred_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    total_errors = fn + fp
    error_rate = total_errors / len(y_test) * 100

    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f'[{name} 평가 결과 (검증셋 기준 Threshold: {best_threshold:.4f}, '
          f'검증 F1: {val_f1:.4f})]')
    print(f'F1-Score            : {f1_val:.4f}')
    print(f'PR-AUC (trapz)      : {pr_auc_val:.4f}')
    print(f'Average Precision   : {ap_val:.4f}')
    print(f'ROC-AUC Score       : {roc_auc_val:.4f}')
    print(f'Precision           : {prec_val:.4f}')
    print(f'Recall              : {rec_val:.4f}')
    print(f'총 오류 건수(FN+FP)  : {total_errors:,}건 '
          f'(미검출 FN: {fn}건, 과잉검출 FP: {fp}건)')
    print(f'오류 비율(%)        : {error_rate:.2f}%')
    print('\n[Classification Report]')
    print(classification_report(y_test, pred, zero_division=0))

    results.append({
        'Model': name,
        'Best Threshold': round(float(best_threshold), 4),
        'F1-Score': f1_val,
        'PR-AUC': pr_auc_val,
        'AP': ap_val,
        'ROC-AUC': roc_auc_val,
        'Precision': prec_val,
        'Recall': rec_val,
        '오류건수(FN+FP)': total_errors,
        '오류비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp,
    })

# ---------------------------------------------------------
# 5. F1-Score 기준 최종 비교 요약 표
# ---------------------------------------------------------
huristic_summary_df = (
    pd.DataFrame(results)
    .sort_values(by='F1-Score', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 70)
print('[전체 모델 종합 성능 비교 요약 (F1-Score 순 정렬)]')
print('=' * 70)
print(huristic_summary_df.to_string(index=False))

  0%|          | 0/6 [00:00<?, ?it/s]


=== LightGBM 모델 학습 및 평가 시작 ===


 17%|█▋        | 1/6 [00:00<00:01,  3.24it/s]

[LightGBM 평가 결과 (검증셋 기준 Threshold: 0.5682, 검증 F1: 0.1212)]
F1-Score            : 0.1333
PR-AUC (trapz)      : 0.0945
Average Precision   : 0.1818
ROC-AUC Score       : 0.7155
Precision           : 0.0800
Recall              : 0.4000
총 오류 건수(FN+FP)  : 52건 (미검출 FN: 6건, 과잉검출 FP: 46건)
오류 비율(%)        : 2.33%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      2226
           1       0.08      0.40      0.13        10

    accuracy                           0.98      2236
   macro avg       0.54      0.69      0.56      2236
weighted avg       0.99      0.98      0.98      2236


=== XGBoost 모델 학습 및 평가 시작 ===


 33%|███▎      | 2/6 [00:01<00:02,  1.39it/s]

...

[CatBoost 평가 결과 (검증셋 기준 Threshold: 0.7249, 검증 F1: 0.1250)]
F1-Score            : 0.3636
PR-AUC (trapz)      : 0.3178
Average Precision   : 0.2319
ROC-AUC Score       : 0.6418
Precision           : 0.3333
Recall              : 0.4000
총 오류 건수(FN+FP)  : 14건 (미검출 FN: 6건, 과잉검출 FP: 8건)
오류 비율(%)        : 0.63%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2226
           1       0.33      0.40      0.36        10

    accuracy                           0.99      2236
   macro avg       0.67      0.70      0.68      2236
weighted avg       0.99      0.99      0.99      2236


=== Random Forest 모델 학습 및 평가 시작 ===


 67%|██████▋   | 4/6 [00:05<00:02,  1.36s/it]

[Random Forest 평가 결과 (검증셋 기준 Threshold: 0.1564, 검증 F1: 0.1429)]
F1-Score            : 0.1860
PR-AUC (trapz)      : 0.3192
Average Precision   : 0.2336
ROC-AUC Score       : 0.7300
Precision           : 0.1212
Recall              : 0.4000
총 오류 건수(FN+FP)  : 35건 (미검출 FN: 6건, 과잉검출 FP: 29건)
오류 비율(%)        : 1.57%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2226
           1       0.12      0.40      0.19        10

    accuracy                           0.98      2236
   macro avg       0.56      0.69      0.59      2236
weighted avg       0.99      0.98      0.99      2236


=== Extra Trees 모델 학습 및 평가 시작 ===


100%|██████████| 6/6 [00:05<00:00,  1.06it/s]

[Extra Trees 평가 결과 (검증셋 기준 Threshold: 0.1412, 검증 F1: 0.1143)]
F1-Score            : 0.1600
PR-AUC (trapz)      : 0.3189
Average Precision   : 0.2334
ROC-AUC Score       : 0.7210
Precision           : 0.1000
Recall              : 0.4000
총 오류 건수(FN+FP)  : 42건 (미검출 FN: 6건, 과잉검출 FP: 36건)
오류 비율(%)        : 1.88%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      2226
           1       0.10      0.40      0.16        10

    accuracy                           0.98      2236
   macro avg       0.55      0.69      0.58      2236
weighted avg       0.99      0.98      0.99      2236


=== Logistic Regression 모델 학습 및 평가 시작 ===
[Logistic Regression 평가 결과 (검증셋 기준 Threshold: 0.0763, 검증 F1: 0.0851)]
F1-Score            : 0.1127
PR-AUC (trapz)      : 0.0755
Average Precision   : 0.1485
ROC-AUC Score       : 0.6028
Precision           : 0.0656
Recall              : 0.4000
총 오류 건수(FN+FP)  : 63건 (미검출 FN: 6건, 과잉검출 FP: 57건)
오류 

In [108]:
huristic_summary_df

,Model,Best Threshold,F1-Score,PR-AUC,AP,ROC-AUC,Precision,Recall,오류건수(FN+FP),오류비율(%),미검출(FN),과잉검출(FP)
0,CatBoost,0.7249,0.363636,0.317764,0.231937,0.641801,0.333333,0.4,14,0.626118,6,8
1,Random Forest,0.1564,0.186047,0.319181,0.233619,0.729987,0.121212,0.4,35,1.565295,6,29
2,Extra Trees,0.1412,0.160000,0.318856,0.233390,0.721024,0.100000,0.4,42,1.878354,6,36
3,LightGBM,0.5682,0.133333,0.094535,0.181821,0.715521,0.080000,0.4,52,2.325581,6,46
4,Logistic Regression,0.0763,0.112676,0.075542,0.148478,0.602785,0.065574,0.4,63,2.817531,6,57
5,XGBoost,0.1326,0.074766,0.089328,0.166867,0.769025,0.041237,0.4,99,4.427549,6,93


### **8. Optuna 튜닝 진행**

In [109]:
!pip install optuna -q

In [114]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, auc, average_precision_score, roc_auc_score,
    confusion_matrix, f1_score, recall_score, precision_score,
)

start = time.perf_counter()
optuna.logging.set_verbosity(optuna.logging.WARNING)

SAVE_DIR = 'best_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------------------------------------------------
# 0. 튜닝/임계값용 검증셋 분리 (테스트셋은 최종 평가에만 사용)
# ---------------------------------------------------------
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_final, y_train,
    test_size=0.2, stratify=y_train, random_state=42,
)


def find_best_threshold(y_true, prob):
    p, r, th = precision_recall_curve(y_true, prob)
    denom = p + r
    f1 = np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)
    best_idx = np.argmax(f1[:-1])  # th 길이에 맞춤
    return th[best_idx], f1[best_idx]


def val_ap(model):
    return average_precision_score(y_val, model.predict_proba(X_val)[:, 1])


# ---------------------------------------------------------
# 1. Objective: 검증셋 Average Precision 최대화
# ---------------------------------------------------------
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params).fit(X_fit, y_fit)
    return val_ap(model)


def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'eval_metric': 'logloss',
    }
    model = XGBClassifier(**params).fit(X_fit, y_fit)
    return val_ap(model)


def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42, 'verbose': 0,
    }
    model = CatBoostClassifier(**params).fit(X_fit, y_fit)
    return val_ap(model)


def objective_rf(trial):
    pos_w = trial.suggest_float('pos_weight', 1.0, 50.0, log=True)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 30),
        'max_features': trial.suggest_float('max_features', 0.05, 0.5),
        'class_weight': {0: 1, 1: pos_w},
        'random_state': 42, 'n_jobs': -1,
    }
    model = RandomForestClassifier(**params).fit(X_fit, y_fit)
    return val_ap(model)


def objective_et(trial):
    pos_w = trial.suggest_float('pos_weight', 1.0, 50.0, log=True)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 30),
        'max_features': trial.suggest_float('max_features', 0.05, 0.5),
        'class_weight': {0: 1, 1: pos_w},
        'random_state': 42, 'n_jobs': -1,
    }
    model = ExtraTreesClassifier(**params).fit(X_fit, y_fit)
    return val_ap(model)


def objective_lr(trial):
    pos_w = trial.suggest_float('pos_weight', 1.0, 50.0, log=True)
    model = make_pipeline(
        StandardScaler(with_mean=False),   # 희소 행렬 대응
        LogisticRegression(
            C=trial.suggest_float('C', 1e-3, 10.0, log=True),
            class_weight={0: 1, 1: pos_w},
            max_iter=2000, random_state=42,
        ),
    ).fit(X_fit, y_fit)
    return val_ap(model)


# ---------------------------------------------------------
# 2. Optuna 튜닝 실행
# ---------------------------------------------------------
N_TRIALS = 30
objectives = {
    'LightGBM': objective_lgb,
    'XGBoost': objective_xgb,
    'CatBoost': objective_cat,
    'Random Forest': objective_rf,
    'Extra Trees': objective_et,
    'Logistic Regression': objective_lr,
}

studies = {}
for name, obj in objectives.items():
    print(f'=== {name} 튜닝 시작 ===')
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(obj, n_trials=N_TRIALS)
    studies[name] = study
    print(f'{name} Best 검증 AP: {study.best_value:.4f}')


# ---------------------------------------------------------
# 3. 최적 파라미터로 모델 생성 함수
# ---------------------------------------------------------
def build_model(name, bp):
    bp = dict(bp)
    if name == 'LightGBM':
        return lgb.LGBMClassifier(**bp, subsample_freq=1,
                                  random_state=42, n_jobs=-1, verbose=-1)
    if name == 'XGBoost':
        return XGBClassifier(**bp, random_state=42, n_jobs=-1,
                             eval_metric='logloss')
    if name == 'CatBoost':
        return CatBoostClassifier(**bp, random_state=42, verbose=0)
    if name == 'Random Forest':
        pos_w = bp.pop('pos_weight')
        return RandomForestClassifier(**bp, class_weight={0: 1, 1: pos_w},
                                      random_state=42, n_jobs=-1)
    if name == 'Extra Trees':
        pos_w = bp.pop('pos_weight')
        return ExtraTreesClassifier(**bp, class_weight={0: 1, 1: pos_w},
                                    random_state=42, n_jobs=-1)

    if name == 'Logistic Regression':
        pos_w = bp.pop('pos_weight')
        return make_pipeline(
            StandardScaler(with_mean=False),   # 희소 행렬 대응
            LogisticRegression(**bp, class_weight={0: 1, 1: pos_w},
                                max_iter=2000, random_state=42),
    )


# ---------------------------------------------------------
# 4. 최종 학습 -> 검증셋 임계값 -> 테스트셋 평가 -> 저장
# ---------------------------------------------------------
tuned_results = []

for name, study in studies.items():
    print(f'\n=== {name} 최적 모델 평가 시작 ===')
    model = build_model(name, study.best_params)
    model.fit(X_fit, y_fit)

    # 임계값은 검증셋에서 결정
    val_prob = model.predict_proba(X_val)[:, 1]
    best_threshold, val_f1 = find_best_threshold(y_val, val_prob)

    # 테스트셋 평가
    pred_prob = model.predict_proba(X_test_final)[:, 1]
    pred = (pred_prob >= best_threshold).astype(int)

    precision_vec, recall_vec, _ = precision_recall_curve(y_test, pred_prob)
    pr_auc_val = auc(recall_vec, precision_vec)
    ap_val = average_precision_score(y_test, pred_prob)
    roc_auc_val = roc_auc_score(y_test, pred_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    total_errors = fn + fp
    error_rate = total_errors / len(y_test) * 100

    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f'[{name} 결과 (검증셋 기준 Threshold: {best_threshold:.4f}, 검증 F1: {val_f1:.4f})]')
    print(f'F1-Score          : {f1_val:.4f}')
    print(f'PR-AUC (trapz)    : {pr_auc_val:.4f}')
    print(f'Average Precision : {ap_val:.4f}')
    print(f'ROC-AUC           : {roc_auc_val:.4f}')
    print(f'Precision         : {prec_val:.4f}')
    print(f'Recall            : {rec_val:.4f}')
    print(f'총 오류 건수(FN+FP): {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP: {fp}건)')

    # ---- 저장 ----
    tag = name.replace(' ', '_').lower()
    joblib.dump(model, os.path.join(SAVE_DIR, f'{tag}_best.joblib'))
    if name == 'CatBoost':
        model.save_model(os.path.join(SAVE_DIR, f'{tag}_best.cbm'))

    with open(os.path.join(SAVE_DIR, f'{tag}_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'model': name,
            'best_params': study.best_params,
            'best_threshold': float(best_threshold),
            'val_AP': float(study.best_value),
            'test_F1': float(f1_val),
            'test_AP': float(ap_val),
        }, f, ensure_ascii=False, indent=2)

    tuned_results.append({
        'Model': name,
        'Best Threshold': round(float(best_threshold), 4),
        'F1-Score': f1_val,
        'PR-AUC': pr_auc_val,
        'AP': ap_val,
        'ROC-AUC': roc_auc_val,
        'Precision': prec_val,
        'Recall': rec_val,
        '오류건수(FN+FP)': total_errors,
        '오류비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp,
    })

# ---------------------------------------------------------
# 5. 결과 요약 (AP 기준 정렬) 및 저장
# ---------------------------------------------------------
tuned_summary_df = (
    pd.DataFrame(tuned_results)
    .sort_values(by='AP', ascending=False)
    .reset_index(drop=True)
)
tuned_summary_df.to_csv(os.path.join(SAVE_DIR, 'tuned_summary.csv'),
                        index=False, encoding='utf-8-sig')

print('\n' + '=' * 70)
print('[Optuna 하이퍼파라미터 튜닝 최종 성능 비교 요약 (AP 순 정렬)]')
print('=' * 70)
print(tuned_summary_df.to_string(index=False))

end = time.perf_counter()
print(f'실행 시간: {end - start:.2f}초')

=== LightGBM 튜닝 시작 ===
LightGBM Best 검증 AP: 0.0724
=== XGBoost 튜닝 시작 ===
XGBoost Best 검증 AP: 0.0434
=== CatBoost 튜닝 시작 ===
CatBoost Best 검증 AP: 0.0517
=== Random Forest 튜닝 시작 ===
Random Forest Best 검증 AP: 0.0696
=== Extra Trees 튜닝 시작 ===
Extra Trees Best 검증 AP: 0.0360
=== Logistic Regression 튜닝 시작 ===
Logistic Regression Best 검증 AP: 0.0316

=== LightGBM 최적 모델 평가 시작 ===
[LightGBM 결과 (검증셋 기준 Threshold: 0.2137, 검증 F1: 0.1818)]
F1-Score          : 0.0000
PR-AUC (trapz)    : 0.0517
Average Precision : 0.0986
ROC-AUC           : 0.6613
Precision         : 0.0000
Recall            : 0.0000
총 오류 건수(FN+FP): 13건 (미검출 FN: 10건, 과잉검출 FP: 3건)

=== XGBoost 최적 모델 평가 시작 ===
[XGBoost 결과 (검증셋 기준 Threshold: 0.1517, 검증 F1: 0.1277)]
F1-Score          : 0.1096
PR-AUC (trapz)    : 0.0843
Average Precision : 0.1640
ROC-AUC           : 0.7004
Precision         : 0.0635
Recall            : 0.4000
총 오류 건수(FN+FP): 65건 (미검출 FN: 6건, 과잉검출 FP: 59건)

=== CatBoost 최적 모델 평가 시작 ===
[CatBoost 결과 (검증셋 기준 Threshold: 0.0170, 

In [115]:
tuned_summary_df

,Model,Best Threshold,F1-Score,PR-AUC,AP,ROC-AUC,Precision,Recall,오류건수(FN+FP),오류비율(%),미검출(FN),과잉검출(FP)
0,Extra Trees,0.1595,0.148148,0.319961,0.234464,0.757951,0.090909,0.4,46,2.057245,6,40
1,XGBoost,0.1517,0.109589,0.084305,0.163959,0.700449,0.063492,0.4,65,2.906977,6,59
2,Logistic Regression,0.0269,0.166667,0.082926,0.163143,0.616981,0.105263,0.4,40,1.788909,6,34
3,CatBoost,0.0170,0.140351,0.080577,0.153858,0.760400,0.085106,0.4,49,2.191413,6,43
4,Random Forest,0.2257,0.000000,0.080105,0.152898,0.747394,0.000000,0.0,13,0.581395,10,3
5,LightGBM,0.2137,0.000000,0.051705,0.098601,0.661298,0.000000,0.0,13,0.581395,10,3


### **tuned_summary_df 분석 결과**
- 기초 선(Baseline) 대비 뛰어난 PR-AUC / AP실제 '검출(Positive)' 비율이 0.313% (0.00313)에 불과
- 무작위 예측 기준선(Baseline)이 0.0031인 상황에서, 최상위 모델인 Extra Trees는 PR-AUC 0.320, AP 0.234를 기록
- 무작위 예측 대비 PR-AUC 기준 약 103배, AP 기준 약 75배 이상 향상된 수치

**Optuna 튜닝으로 개선된 핵심 원인(정확한 내용은 아님)**
- Precision과 Recall의 균형점 확보:휴리스틱에서는 Threshold를 0.9 이상으로 끌어올려 강제로 FP를 눌렀기 때문에 Recall이 떨어졌지만, Optuna가 scale_pos_weight와 트리 깊이를 동시에 최적화하면서 Recall(21~23%)을 유지한 채 Precision(20%)을 함께 끌어올렸습니다.   
- Best Threshold의 정상화:LightGBM 기준 Threshold가 0.93에서 0.14로 내려왔습니다. 이는 모델이 예측하는 확률값 분포 자체가 훨씬 신뢰할 수 있게 보정(Calibration)되었음을 의미합니다.   
- 오류 건수 감소:CatBoost의 경우 총 오류 건수가 678건에서 591건으로 줄어들며 가장 적은 오류를 기록했습니다.

### **9. Stacking Model after Optuna Tunned models**

- Optuna로 최적화된 3개 모델(LightGBM, XGBoost, CatBoost)의 Out-of-Fold(OOF) 예측 확률을 메타 피처(Meta-feature)로 생성하고, 이를 바탕으로 최종 Meta-Learner(Logistic Regression)를 학습시키는 스태킹(Stacking) 앙상블 코드

- 데이터 누수(Data Leakage)를 방지하기 위해 Stratified K-Fold 기반의 OOF 방식을 적용

In [116]:
import json
import joblib
import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix,
    f1_score, precision_score, recall_score,
)

SAVE_DIR = 'best_models'
stack_names = ['Extra Trees', 'CatBoost', 'XGBoost', 'Logistic Regression']
tag = lambda n: n.replace(' ', '_').lower()

# ---------------------------------------------------------
# 1. 저장된 base 모델 불러오기
# ---------------------------------------------------------
base = {n: joblib.load(f'{SAVE_DIR}/{tag(n)}_best.joblib') for n in stack_names}


def to_logit(p):
    # 모델마다 확률 스케일(class_weight 등)이 달라서 logit으로 맞춤
    return logit(np.clip(p, 1e-6, 1 - 1e-6))


def base_features(X):
    return np.column_stack(
        [to_logit(m.predict_proba(X)[:, 1]) for m in base.values()]
    )


# 메타 모델 학습용: base 모델이 보지 않은 검증셋
Z_val = base_features(X_val)
Z_test = base_features(X_test_final)

print('[base 모델 검증셋 AP]')
for i, n in enumerate(stack_names):
    print(f'  {n:20s}: {average_precision_score(y_val, Z_val[:, i]):.4f}')

# ---------------------------------------------------------
# 2. 메타 모델 (Logistic Regression) - 검증셋 5-fold OOF로 성능/임계값 산출
# ---------------------------------------------------------
meta = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = cross_val_predict(meta, Z_val, y_val, cv=skf, method='predict_proba')[:, 1]
print(f'\n[Stacking 검증 OOF AP]: {average_precision_score(y_val, oof):.4f}')

best_thr, val_f1 = find_best_threshold(y_val, oof)

# 비교용 baseline: logit 단순 평균
avg_val = Z_val.mean(axis=1)
avg_test = Z_test.mean(axis=1)
print(f'[단순 평균 검증 AP]      : {average_precision_score(y_val, avg_val):.4f}')

# 최종 메타 모델은 검증셋 전체로 학습
meta.fit(Z_val, y_val)
print('\n[메타 모델 계수]')
for n, c in zip(stack_names, meta.coef_[0]):
    print(f'  {n:20s}: {c:+.3f}')

# ---------------------------------------------------------
# 3. 테스트셋 평가
# ---------------------------------------------------------
def evaluate(name, prob, thr):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        'Model': name,
        'AP': average_precision_score(y_test, prob),
        'ROC-AUC': roc_auc_score(y_test, prob),
        'F1': f1_score(y_test, pred, zero_division=0),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'FN': fn, 'FP': fp,
    }


stack_test_prob = meta.predict_proba(Z_test)[:, 1]
avg_thr, _ = find_best_threshold(y_val, avg_val)

rows = [
    evaluate('Stacking (meta LR)', stack_test_prob, best_thr),
    evaluate('Logit 평균', avg_test, avg_thr),
]
# 개별 모델과 비교
for i, n in enumerate(stack_names):
    thr_i, _ = find_best_threshold(y_val, Z_val[:, i])
    rows.append(evaluate(n, Z_test[:, i], thr_i))

print('\n' + pd.DataFrame(rows).to_string(index=False))

# ---------------------------------------------------------
# 4. 스태킹 모델 저장 및 새 데이터 예측 함수
# ---------------------------------------------------------
joblib.dump(
    {'base_names': stack_names, 'meta': meta, 'threshold': float(best_thr)},
    f'{SAVE_DIR}/stacking_final.joblib',
)


def predict_stack(X_new):
    bundle = joblib.load(f'{SAVE_DIR}/stacking_final.joblib')
    models = [joblib.load(f'{SAVE_DIR}/{tag(n)}_best.joblib')
              for n in bundle['base_names']]
    Z = np.column_stack(
        [to_logit(m.predict_proba(X_new)[:, 1]) for m in models]
    )
    prob = bundle['meta'].predict_proba(Z)[:, 1]
    return prob, (prob >= bundle['threshold']).astype(int)

[base 모델 검증셋 AP]
  Extra Trees         : 0.0360
  CatBoost            : 0.0517
  XGBoost             : 0.0434
  Logistic Regression : 0.0316

[Stacking 검증 OOF AP]: 0.0141
[단순 평균 검증 AP]      : 0.0418

[메타 모델 계수]
  Extra Trees         : +0.632
  CatBoost            : +2.250
  XGBoost             : -0.373
  Logistic Regression : +0.230

              Model       AP  ROC-AUC       F1  Precision  Recall  FN  FP
 Stacking (meta LR) 0.130315 0.727987 0.195122   0.129032     0.4   6  27
           Logit 평균 0.182513 0.698248 0.177778   0.114286     0.4   6  31
        Extra Trees 0.234464 0.757951 0.148148   0.090909     0.4   6  40
           CatBoost 0.153858 0.760400 0.140351   0.085106     0.4   6  43
            XGBoost 0.163959 0.700449 0.109589   0.063492     0.4   6  59
Logistic Regression 0.163143 0.616981 0.166667   0.105263     0.4   6  34


In [117]:
pd.DataFrame(rows)

,Model,AP,ROC-AUC,F1,Precision,Recall,FN,FP
0,Stacking (meta LR),0.130315,0.727987,0.195122,0.129032,0.4,6,27
1,Logit 평균,0.182513,0.698248,0.177778,0.114286,0.4,6,31
2,Extra Trees,0.234464,0.757951,0.148148,0.090909,0.4,6,40
3,CatBoost,0.153858,0.760400,0.140351,0.085106,0.4,6,43
4,XGBoost,0.163959,0.700449,0.109589,0.063492,0.4,6,59
5,Logistic Regression,0.163143,0.616981,0.166667,0.105263,0.4,6,34


### **10. Stacking Model with Optuna Tunning**

- **모델을 각각 Optuna로 튜닝하고 Staking Model 적용**

In [ ]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from scipy.special import logit
from scipy.stats import rankdata
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict,
)
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, roc_auc_score,
    confusion_matrix, f1_score, precision_score, recall_score,
)

t0 = time.perf_counter()
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# 0. 설정
# ---------------------------------------------------------
SEED = 42
N_TRIALS = 30          # 튜닝 시도 횟수 (모델당)
TUNE_SIZE = 0.4        # 튜닝용 비율 (나머지 60%는 스태킹용)
TUNE_CV = 3            # 튜닝 시 CV fold 수
STACK_CV = 5           # OOF 생성 fold 수
SAVE_DIR = 'stacking_best_models'
os.makedirs(SAVE_DIR, exist_ok=True)

ALL_MODELS = ['LightGBM', 'XGBoost', 'CatBoost',
              'Random Forest', 'Extra Trees', 'Logistic Regression']
DIVERSE_4 = ['Extra Trees', 'CatBoost', 'XGBoost', 'Logistic Regression']

y_all = np.asarray(y_train)
y_te = np.asarray(y_test)


def take(X, idx):
    return X.iloc[idx] if hasattr(X, 'iloc') else X[idx]


def to_logit(p):
    # 모델마다 class_weight 때문에 확률 스케일이 달라서 logit으로 맞춤
    return logit(np.clip(p, 1e-6, 1 - 1e-6))


def find_best_threshold(y_true, score):
    p, r, th = precision_recall_curve(y_true, score)
    d = p + r
    f1 = np.divide(2 * p * r, d, out=np.zeros_like(d), where=d > 0)
    i = np.argmax(f1[:-1])
    return th[i], f1[i]


# ---------------------------------------------------------
# 1. 데이터 3분할: 튜닝용 / 스태킹용 / 테스트(최종 평가)
# ---------------------------------------------------------
X_tune, X_stack, y_tune, y_stack = train_test_split(
    X_train_final, y_all, train_size=TUNE_SIZE,
    stratify=y_all, random_state=SEED,
)
print(f'튜닝용 {len(y_tune):,}건 (양성 {y_tune.sum():,}) | '
      f'스태킹용 {len(y_stack):,}건 (양성 {y_stack.sum():,}) | '
      f'테스트 {len(y_te):,}건 (양성 {y_te.sum():,})')


# ---------------------------------------------------------
# 2. 모델별 탐색 공간 / 모델 생성
# ---------------------------------------------------------
def suggest_params(name, t):
    if name == 'LightGBM':
        return {
            'n_estimators': t.suggest_int('n_estimators', 100, 500),
            'learning_rate': t.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': t.suggest_int('max_depth', 3, 10),
            'num_leaves': t.suggest_int('num_leaves', 15, 127),
            'min_child_samples': t.suggest_int('min_child_samples', 20, 200),
            'subsample': t.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': t.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_lambda': t.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'scale_pos_weight': t.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        }
    if name == 'XGBoost':
        return {
            'n_estimators': t.suggest_int('n_estimators', 100, 500),
            'learning_rate': t.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': t.suggest_int('max_depth', 3, 10),
            'min_child_weight': t.suggest_int('min_child_weight', 2, 30),
            'subsample': t.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': t.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_lambda': t.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'scale_pos_weight': t.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        }
    if name == 'CatBoost':
        return {
            'iterations': t.suggest_int('iterations', 100, 500),
            'learning_rate': t.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': t.suggest_int('depth', 3, 10),
            'l2_leaf_reg': t.suggest_float('l2_leaf_reg', 1.0, 20.0, log=True),
            'scale_pos_weight': t.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        }
    if name in ('Random Forest', 'Extra Trees'):
        return {
            'n_estimators': t.suggest_int('n_estimators', 100, 400),
            'max_depth': t.suggest_int('max_depth', 4, 25),
            'min_samples_leaf': t.suggest_int('min_samples_leaf', 1, 50),
            'max_features': t.suggest_float('max_features', 0.02, 0.5),
            'pos_weight': t.suggest_float('pos_weight', 1.0, 50.0, log=True),
        }
    if name == 'Logistic Regression':
        return {
            'C': t.suggest_float('C', 1e-3, 10.0, log=True),
            'pos_weight': t.suggest_float('pos_weight', 1.0, 50.0, log=True),
        }


def make_model(name, params):
    p = dict(params)
    if name == 'LightGBM':
        return lgb.LGBMClassifier(**p, subsample_freq=1, random_state=SEED,
                                  n_jobs=-1, verbose=-1)
    if name == 'XGBoost':
        return XGBClassifier(**p, tree_method='hist', random_state=SEED,
                             n_jobs=-1, eval_metric='logloss')
    if name == 'CatBoost':
        return CatBoostClassifier(**p, random_state=SEED, verbose=0)
    if name == 'Random Forest':
        w = p.pop('pos_weight')
        return RandomForestClassifier(**p, class_weight={0: 1, 1: w},
                                      random_state=SEED, n_jobs=-1)
    if name == 'Extra Trees':
        w = p.pop('pos_weight')
        return ExtraTreesClassifier(**p, class_weight={0: 1, 1: w},
                                    random_state=SEED, n_jobs=-1)
    if name == 'Logistic Regression':
        w = p.pop('pos_weight')
        return make_pipeline(
            StandardScaler(with_mean=False),   # 희소 행렬 대응
            LogisticRegression(**p, class_weight={0: 1, 1: w},
                               max_iter=2000, random_state=SEED),
        )


# ---------------------------------------------------------
# 3. Optuna 튜닝: 튜닝용 데이터에서만, CV 평균 AP 최대화
# ---------------------------------------------------------
tune_folds = list(StratifiedKFold(TUNE_CV, shuffle=True, random_state=SEED)
                  .split(np.zeros(len(y_tune)), y_tune))


def make_objective(name):
    def objective(trial):
        params = suggest_params(name, trial)
        scores = []
        for k, (tr, va) in enumerate(tune_folds):
            m = make_model(name, params).fit(take(X_tune, tr), y_tune[tr])
            prob = m.predict_proba(take(X_tune, va))[:, 1]
            scores.append(average_precision_score(y_tune[va], prob))
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective


studies = {}
for name in ALL_MODELS:
    print(f'=== {name} 튜닝 시작 ===')
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(make_objective(name), n_trials=N_TRIALS)
    studies[name] = study
    print(f'  튜닝 CV AP: {study.best_value:.4f}')

with open(f'{SAVE_DIR}/best_params.json', 'w', encoding='utf-8') as f:
    json.dump({n: s.best_params for n, s in studies.items()},
              f, ensure_ascii=False, indent=2)

# ---------------------------------------------------------
# 4. 스태킹용 데이터에서 OOF 예측 생성 (모든 모델이 같은 fold 사용)
# ---------------------------------------------------------
stack_folds = list(StratifiedKFold(STACK_CV, shuffle=True, random_state=SEED + 1)
                   .split(np.zeros(len(y_stack)), y_stack))

oof_z = np.zeros((len(y_stack), len(ALL_MODELS)))
test_z = np.zeros((len(y_te), len(ALL_MODELS)))
fold_models = {n: [] for n in ALL_MODELS}

for j, name in enumerate(ALL_MODELS):
    print(f'=== {name} OOF 생성 ===')
    for tr, va in stack_folds:
        m = make_model(name, studies[name].best_params)
        m.fit(take(X_stack, tr), y_stack[tr])
        oof_z[va, j] = to_logit(m.predict_proba(take(X_stack, va))[:, 1])
        test_z[:, j] += to_logit(m.predict_proba(X_test_final)[:, 1]) / STACK_CV
        fold_models[name].append(m)
    print(f'  OOF AP: {average_precision_score(y_stack, oof_z[:, j]):.4f}')

# ---------------------------------------------------------
# 5. 메타 모델: OOF 피처로 학습, 성능은 한 번 더 교차검증으로 측정
# ---------------------------------------------------------
def run_stack(model_list, C=0.5):
    cols = [ALL_MODELS.index(n) for n in model_list]
    meta = LogisticRegression(C=C, class_weight='balanced', max_iter=1000)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED + 2)
    meta_oof = cross_val_predict(meta, oof_z[:, cols], y_stack, cv=cv,
                                 method='predict_proba')[:, 1]
    meta.fit(oof_z[:, cols], y_stack)
    return meta, cols, meta_oof, meta.predict_proba(test_z[:, cols])[:, 1]


meta_all, cols_all, oof_all, te_all = run_stack(ALL_MODELS)
meta_div, cols_div, oof_div, te_div = run_stack(DIVERSE_4)

print('\n[메타 계수: 6개 전체]')
for n, c in zip(ALL_MODELS, meta_all.coef_[0]):
    print(f'  {n:20s}: {c:+.3f}')


def rank_norm(v):
    return rankdata(v) / len(v)


def rank_mean(z):
    return np.mean([rank_norm(z[:, j]) for j in range(z.shape[1])], axis=0)


candidates = {
    'Stacking (6개 전체)': (oof_all, te_all),
    'Stacking (다양성 4개)': (oof_div, te_div),
    'Logit 평균 (6개)': (oof_z.mean(axis=1), test_z.mean(axis=1)),
    '순위 평균 (6개)': (rank_mean(oof_z), rank_mean(test_z)),
}
for j, n in enumerate(ALL_MODELS):
    candidates[n] = (oof_z[:, j], test_z[:, j])

# ---------------------------------------------------------
# 6. 평가: 임계값은 OOF에서 결정, 테스트셋에는 그대로 적용
# ---------------------------------------------------------
rows, thresholds = [], {}
for cname, (s_oof, s_test) in candidates.items():
    thr, oof_f1 = find_best_threshold(y_stack, s_oof)
    thresholds[cname] = thr
    pred = (s_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    rows.append({
        'Model': cname,
        'OOF_AP': average_precision_score(y_stack, s_oof),
        'OOF_ROC': roc_auc_score(y_stack, s_oof),
        'OOF_F1': oof_f1,
        'Test_AP': average_precision_score(y_te, s_test),
        'Test_ROC': roc_auc_score(y_te, s_test),
        'Test_F1': f1_score(y_te, pred, zero_division=0),
        'Precision': precision_score(y_te, pred, zero_division=0),
        'Recall': recall_score(y_te, pred, zero_division=0),
        'FN': fn, 'FP': fp,
    })

result_df = (pd.DataFrame(rows).sort_values('OOF_AP', ascending=False)
             .reset_index(drop=True))
result_df.to_csv(f'{SAVE_DIR}/stacking_comparison.csv',
                 index=False, encoding='utf-8-sig')
print('\n[OOF AP 순 정렬]')
print(result_df.round(4).to_string(index=False))

# ---------------------------------------------------------
# 7. 저장 및 예측 함수 (스태킹 6개 전체 기준; 다른 조합은 아래 주석 참고)
# ---------------------------------------------------------
joblib.dump(
    {
        'names': ALL_MODELS, 'cols': cols_all, 'fold_models': fold_models,
        'meta': meta_all,
        'threshold': float(thresholds['Stacking (6개 전체)']),
    },
    f'{SAVE_DIR}/stacking_final.joblib', compress=3,
)


def predict_stack(X_new, path=f'{SAVE_DIR}/stacking_final.joblib'):
    b = joblib.load(path)
    Z = np.column_stack([
        np.mean([to_logit(m.predict_proba(X_new)[:, 1])
                 for m in b['fold_models'][n]], axis=0)
        for n in b['names']
    ])
    prob = b['meta'].predict_proba(Z[:, b['cols']])[:, 1]
    return prob, (prob >= b['threshold']).astype(int)

# 다양성 4개 조합을 저장하려면 위 dump에서
# 'cols': cols_div, 'meta': meta_div, 'threshold': thresholds['Stacking (다양성 4개)']
# 로 바꾸면 됩니다.

print(f'\n실행 시간: {time.perf_counter() - t0:.1f}초')

튜닝용 3,576건 (양성 15) | 스태킹용 5,366건 (양성 23) | 테스트 2,236건 (양성 10)
=== LightGBM 튜닝 시작 ===
  튜닝 CV AP: 0.1510
=== XGBoost 튜닝 시작 ===
  튜닝 CV AP: 0.0091
=== CatBoost 튜닝 시작 ===


In [ ]:
result_df

## **[11. Autogluon](https://auto.gluon.ai/stable/index.html)**
- stacking 모델 패키지로 Tunnig 해 보기